# Imports

In [1]:
import numpy as np
import pandas as pd
import requests

C:\Users\ludov\miniconda3\envs\env39\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# Appels API

## Liste clients

In [2]:
def get_clients_list(base_url):
    """
    Récupère la liste des IDs clients disponibles.

    Args:
        base_url (str): URL de base de l'API (ex: "https://ton-api-render.onrender.com")

    Returns:
        list: Liste des IDs clients ou un dictionnaire d'erreur.
    """
    try:
        response = requests.get(f"{base_url}/clients_list")
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        return {"error": f"Erreur HTTP : {err}"}
    except requests.exceptions.RequestException as err:
        return {"error": f"Erreur de requête : {err}"}

base_url = "https://openclassrooms-datascientist-projet7.onrender.com/"
clients_list = get_clients_list(base_url)

if isinstance(clients_list, list):
    print("Liste des IDs clients :", clients_list[:5], "...") 
else:
    print("Erreur :", clients_list["error"])

Liste des IDs clients : [100001, 100005, 100013, 100028, 100038] ...


## Probabilité d'échec

In [3]:
def get_predicted_failure_rate(base_url, client_id, path_to_threshold="useful_saved_parameters.csv"):
    """
    Récupère la probabilité de défaut pour un client donné et prend une décision (Accepté/Refusé) en fonction d'un seuil.

    Args:
        base_url (str): URL de base de l'API (ex: "https://openclassrooms-datascientist-projet7.onrender.com/")
        client_id (int): ID du client
        path_to_threshold (str): Chemin vers le fichier CSV contenant le seuil

    Returns:
        dict: Dictionnaire contenant la probabilité et la décision, ou un dictionnaire d'erreur.
    """
    try:
        threshold_df = pd.read_csv(path_to_threshold)
        threshold = threshold_df.iloc[0, 0]

        response = requests.get(f"{base_url}/predict_proba", params={"id": client_id})
        response.raise_for_status()
        result = response.json()
        predicted_failure_rate = result["predicted_failure_rate"]
        client_proba = np.round(predicted_failure_rate * 100, 1)

        decision = "Accepté" if client_proba < threshold else "Refusé"

        return {
            "client_proba": client_proba,
            "decision": decision,
            "threshold": threshold
        }
    except FileNotFoundError:
        return {"error": f"Fichier '{path_to_threshold}' introuvable."}
    except pd.errors.EmptyDataError:
        return {"error": "Le fichier CSV est vide ou mal formaté."}
    except requests.exceptions.HTTPError as err:
        return {"error": f"Erreur HTTP : {err}"}
    except requests.exceptions.RequestException as err:
        return {"error": f"Erreur de requête : {err}"}
    except Exception as err:
        return {"error": f"Erreur inattendue : {err}"}

base_url = "https://openclassrooms-datascientist-projet7.onrender.com/"
client_id = 100001
result = get_predicted_failure_rate(base_url, client_id)

if "client_proba" in result:
    print(f"Probabilité de défaut pour le client {client_id} : {result['client_proba']:.1f}%")
    print(f"Seuil de décision : {result['threshold']:.1f}%")
    print(f"Décision : {result['decision']}")
else:
    print(f"Erreur : {result['error']}")

Probabilité de défaut pour le client 100001 : 50.2%
Seuil de décision : 53.1%
Décision : Accepté


## Features

In [4]:
def get_client_features(base_url, client_id):
    """
    Récupère les features brutes d'un client.

    Args:
        base_url (str): URL de base de l'API
        client_id (int): ID du client

    Returns:
        list: Liste des features ou un dictionnaire d'erreur.
    """
    try:
        response = requests.get(f"{base_url}/client_features", params={"id": client_id})
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        return {"error": f"Erreur HTTP : {err}"}
    except requests.exceptions.RequestException as err:
        return {"error": f"Erreur de requête : {err}"}

features = get_client_features(base_url, client_id)

if isinstance(features, list):
    print(f"Features du client {client_id} :", features[:5], "...")  
else:
    print("Erreur :", features["error"])

Features du client 100001 : [[0, 0, 1, 0, 135000.0, 568800.0, 20560.5, 0.01885, -19241, -2329, -5170.0, -812, 1, 1, 0, 1, 0, 1, 2, 18, 0, 0, 0, 0, 0.7896543511176771, 0.1595195404777181, 0.0, 0.0, -1740.0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, False, False, False, False, False, False, True, False, False, False, False, False, True, False, True, False, False, False, True, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, Fals

## Features explainer

In [5]:
def get_client_features_prep(base_url, client_id):
    """
    Récupère les features préparées pour l'explainer.

    Args:
        base_url (str): URL de base de l'API
        client_id (int): ID du client

    Returns:
        list: Liste des features préparées ou un dictionnaire d'erreur.
    """
    try:
        response = requests.get(f"{base_url}/client_features_prep", params={"id": client_id})
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        return {"error": f"Erreur HTTP : {err}"}
    except requests.exceptions.RequestException as err:
        return {"error": f"Erreur de requête : {err}"}

prep_features = get_client_features_prep(base_url, client_id)

if isinstance(prep_features, list):
    print(f"Features préparées pour le client {client_id} :", prep_features[:5], "...")  
else:
    print("Erreur :", prep_features["error"])

Features préparées pour le client 100001 : [[0.0, 0.0, 1.0, 0.0, 0.000934820325994545, 0.13151947934556632, 0.08325766289323405, 0.2544911427579021, -1.5363584474885843, 0.8613016763109229, 0.7792107960368979, 0.8695163104611924, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.5, 0.7826086956521738, 0.0, 0.0, 0.0, 0.0, 0.9235718126199157, 0.17754932513386565, 0.0, 0.0, 0.5784883720930233, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

## Clients similaires

In [6]:
def get_similar_clients(base_url, client_id, features):
    """
    Récupère les features des clients similaires à un client donné.

    Args:
        base_url (str): URL de base de l'API
        client_id (int): ID du client
        features (list): Liste des features à retourner pour les clients similaires

    Returns:
        dict: Dictionnaire des features des clients similaires ou un dictionnaire d'erreur.
    """
    try:
        data = {"id": client_id, "features": features}
        response = requests.post(f"{base_url}/similar_clients", json=data)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        return {"error": f"Erreur HTTP : {err}"}
    except requests.exceptions.RequestException as err:
        return {"error": f"Erreur de requête : {err}"}

features = ["DAYS_BIRTH", "AMT_INCOME_TOTAL", "AMT_CREDIT"]
similar_clients = get_similar_clients(base_url, client_id, features)

if isinstance(similar_clients, dict) and "error" not in similar_clients:
    print(f"Features des clients similaires au client {client_id} :")
    for client_index, client_data in similar_clients.items():
        print(f"Client {client_index}:")
        for feature, value in client_data.items():
            print(f"  {feature}: {value}")
else:
    print("Erreur :", similar_clients.get("error", "Réponse inattendue"))

Features des clients similaires au client 100001 :
Client AMT_CREDIT:
  0: 568800.0
  1: 540000.0
  2: 594121.5
  3: 539100.0
  4: 625500.0
  5: 539100.0
  6: 499221.0
  7: 499221.0
  8: 490374.0
  9: 544500.0
  10: 478498.5
Client AMT_INCOME_TOTAL:
  0: 135000.0
  1: 135000.0
  2: 157500.0
  3: 157500.0
  4: 180000.0
  5: 211500.0
  6: 180000.0
  7: 90000.0
  8: 180000.0
  9: 229500.0
  10: 175500.0
Client DAYS_BIRTH:
  0: -19241.0
  1: -22784.0
  2: -10015.0
  3: -13172.0
  4: -13040.0
  5: -13900.0
  6: -16685.0
  7: -22705.0
  8: -12994.0
  9: -12567.0
  10: -17955.0
